# 03 · Word2Vec / GloVe + MLP, and BiLSTM

Phases 4–5. Turn on the GPU first (Runtime → Change runtime type → T4 GPU). If installing `gensim` breaks NumPy, restart the runtime (Runtime → Restart session) and run the cells again from the top.

## Setup
Clones the repo, installs packages, and links `models/` and `mlruns/` to Google Drive so nothing is lost if Colab disconnects.

In [ ]:
# 1. Get the code (for a private repo, see the README: add GITHUB_TOKEN in Colab Secrets)
import os
if not os.path.exists("/content/fineguard"):
    try:
        from google.colab import userdata
        token = userdata.get("GITHUB_TOKEN")
        !git clone -q https://{token}@github.com/424103-achu/fineguard.git /content/fineguard
    except Exception:
        !git clone -q https://github.com/424103-achu/fineguard.git /content/fineguard
%cd /content/fineguard/ml
!git pull -q
!pip install -q -r requirements.txt

In [ ]:
# 2. Save outputs to Google Drive so they survive a disconnect
from google.colab import drive
drive.mount("/content/drive")
DRIVE = "/content/drive/MyDrive/fineguard"
!mkdir -p {DRIVE}/models {DRIVE}/mlruns {DRIVE}/figures
!rm -rf models mlruns && ln -sfn {DRIVE}/models models && ln -sfn {DRIVE}/mlruns mlruns

# MLflow's SQLite file works best on local disk: copy it from Drive now, back at the end
import os, shutil
os.environ["MLFLOW_TRACKING_URI"] = "sqlite:////content/mlflow.db"
os.environ["MLFLOW_DISABLE_AGENT_HINT"] = "1"
if os.path.exists(f"{DRIVE}/mlruns/mlflow.db"):
    shutil.copy(f"{DRIVE}/mlruns/mlflow.db", "/content/mlflow.db")
FIG = f"{DRIVE}/figures"

## MLP on averaged embeddings
Domain-trained Word2Vec (on the 5.5K training sentences) vs general-purpose GloVe.

In [ ]:
!python -m src.train_neural --model mlp --emb w2v --seeds 42 43 44
!python -m src.train_neural --model mlp --emb glove --seeds 42 43 44

In [ ]:
# Copy the MLflow database back to Drive (run this after every training cell you care about)
shutil.copy("/content/mlflow.db", f"{DRIVE}/mlruns/mlflow.db")
!python -m src.results_table --out {DRIVE}/results.md

## BiLSTM
Main run, then one row each for frozen embeddings and last-hidden-state pooling.

In [ ]:
!python -m src.train_neural --model bilstm --emb w2v --seeds 42 43 44
!python -m src.train_neural --model bilstm --emb glove --seeds 42 43 44

In [ ]:
!python -m src.train_neural --model bilstm --emb w2v --freeze-emb --seeds 42
!python -m src.train_neural --model bilstm --emb w2v --pooling last --seeds 42

In [ ]:
# Copy the MLflow database back to Drive (run this after every training cell you care about)
shutil.copy("/content/mlflow.db", f"{DRIVE}/mlruns/mlflow.db")
!python -m src.results_table --out {DRIVE}/results.md